# Chess Tutor — Advanced Experiments
**STA561D: Probabilistic Machine Learning**

Three experiments probing system behaviour beyond surface readability metrics.

| Experiment | Question | Method |
|------------|----------|--------|
| **1. ELO Recovery** | Can the system's own explanations be classified back to their target ELO? | Feed explanations to Claude as a classifier. Measures semantic depth of calibration. |
| **2. Strategy-ELO Interaction** | Does strategy injection produce more differentiation at higher ELO? | TF-IDF cosine similarity between Aggressive vs Positional explanations across ELO bands. |
| **3. Positional Complexity** | Does explanation faithfulness degrade on harder positions? | Three position types (opening / middlegame / endgame) × four ELO levels. Faithfulness scored by piece/square reference accuracy. |

## Setup

In [ ]:
import os, sys, asyncio, re, json
import chess, chess.svg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import anthropic
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import SVG, display
from dotenv import load_dotenv

load_dotenv()

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

ELO_LEVELS = [800, 1200, 1600, 1800]
ELO_LABELS = {800: 'Beginner (800)', 1200: 'Intermediate (1200)',
               1600: 'Club Player (1600)', 1800: 'Advanced (1800)'}

# Reuse structured prompt builder from tutor.py
from tutor import get_move_explanation, STRATEGY_PROFILES
from engine import get_best_move

def call_api(prompt, max_tokens=500):
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return msg.content[0].text.strip()

print('Setup complete.')

---
# Experiment 1: ELO Recovery
**Hypothesis:** If the explanations are genuinely semantically differentiated by ELO, a language model should be able to classify them back to the correct ELO band without seeing the original prompt.

**Design:** Generate structured explanations at all four ELO levels. Blind the ELO label. Ask Claude to classify each explanation into one of four ELO bands. Measure classification accuracy and confusion.

**Why this is a stricter test than readability scores:** Readability metrics measure syllable counts and sentence length — surface features. ELO recovery tests whether the *conceptual content and vocabulary* is semantically distinct enough to be identified.

In [ ]:
# Generate structured explanations for the reference position
TEST_FEN  = 'r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/2N2N2/PPPP1PPP/R1BQK2R w KQkq - 4 5'
MOVE_SAN  = 'O-O'
MOVE_UCI  = 'e1g1'
EVAL_STR  = '+0.19'

board = chess.Board(TEST_FEN)

print('Generating structured explanations for ELO recovery test...')
exp1_explanations = {}
for elo in ELO_LEVELS:
    print(f'  ELO {elo}...', end=' ', flush=True)
    text = get_move_explanation(
        board=board, move_san=MOVE_SAN, move_uci=MOVE_UCI,
        elo=elo, evaluation=EVAL_STR, strategy='Balanced'
    )
    exp1_explanations[elo] = text
    print(f'done ({len(text.split())} words)')

print('\nAll explanations generated.')

In [ ]:
# Classifier prompt — ask Claude to identify ELO from explanation alone
CLASSIFIER_SYSTEM = """You are an expert chess coach and educator.
You will be shown a chess move explanation written for a player of unknown ELO rating.
Your task is to identify which ELO band the explanation was written for.

ELO bands to choose from:
- 800  (complete beginner: very simple language, no chess jargon, basic concepts only)
- 1200 (casual player: basic chess terms, simple tactics, 2-3 move thinking)
- 1600 (club player: full chess vocabulary, positional concepts, 4-5 move plans)
- 1800 (advanced: technical language, strategic depth, opponent counterplay discussed)

Respond with ONLY a JSON object in this exact format:
{"predicted_elo": <one of 800, 1200, 1600, 1800>, "confidence": <"high", "medium", or "low">, "reasoning": "<one sentence>"}"""

print('Running ELO classifier on each explanation...')
recovery_results = []

for true_elo in ELO_LEVELS:
    explanation = exp1_explanations[true_elo]
    prompt = f"{CLASSIFIER_SYSTEM}\n\nExplanation to classify:\n{explanation}"

    print(f'  Classifying ELO {true_elo} explanation...', end=' ', flush=True)
    raw = call_api(prompt, max_tokens=200)

    try:
        # Extract JSON from response
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        parsed = json.loads(json_match.group()) if json_match else {}
        predicted = int(parsed.get('predicted_elo', -1))
        confidence = parsed.get('confidence', 'unknown')
        reasoning = parsed.get('reasoning', raw[:100])
    except Exception:
        predicted, confidence, reasoning = -1, 'parse_error', raw[:100]

    correct = (predicted == true_elo)
    recovery_results.append({
        'True ELO': true_elo,
        'Predicted ELO': predicted,
        'Correct': correct,
        'Confidence': confidence,
        'Reasoning': reasoning
    })
    print(f'predicted={predicted} | {"CORRECT" if correct else "WRONG"} | {confidence}')

df_recovery = pd.DataFrame(recovery_results)
accuracy = df_recovery['Correct'].mean()
print(f'\nOverall accuracy: {accuracy:.0%} ({df_recovery["Correct"].sum()}/{len(ELO_LEVELS)})')

In [ ]:
print('ELO RECOVERY RESULTS')
print('='*70)
for _, row in df_recovery.iterrows():
    status = '✓' if row['Correct'] else '✗'
    print(f"{status} True: {row['True ELO']:4d} | Predicted: {row['Predicted ELO']:4d} | "
          f"Confidence: {row['Confidence']:6s} | {row['Reasoning']}")
print(f"\nAccuracy: {accuracy:.0%}")
print()

# Confusion analysis: how far off are wrong predictions?
df_recovery['ELO Error'] = abs(df_recovery['True ELO'] - df_recovery['Predicted ELO'])
print(f"Mean absolute ELO error: {df_recovery['ELO Error'].mean():.0f} ELO points")
print(f"Max error: {df_recovery['ELO Error'].max():.0f} ELO points")
print()
print('INTERPRETATION:')
if accuracy >= 0.75:
    print('Strong recovery — explanations are semantically distinct across ELO bands.')
elif accuracy >= 0.5:
    print('Partial recovery — some ELO bands are well-differentiated, others overlap.')
else:
    print('Weak recovery — surface readability differs but semantic depth is less distinct.')
    print('This suggests vocabulary constraints alone are insufficient for deep calibration.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#2ecc71' if r['Correct'] else '#e74c3c' for _, r in df_recovery.iterrows()]
bars = ax.bar(
    [f"True: {r['True ELO']}\nPredicted: {r['Predicted ELO']}" for _, r in df_recovery.iterrows()],
    [1] * len(df_recovery),
    color=colors, width=0.5
)

for bar, (_, row) in zip(bars, df_recovery.iterrows()):
    label = f"{row['Confidence'].upper()}"
    ax.text(bar.get_x() + bar.get_width()/2, 0.5, label,
            ha='center', va='center', fontweight='bold', fontsize=11, color='white')

ax.set_yticks([])
ax.set_title(f'ELO Recovery Test — Accuracy: {accuracy:.0%}\n(Green = correct classification, Red = misclassified)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Classification Result', fontsize=11)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2ecc71', label='Correct'),
    Patch(color='#e74c3c', label='Misclassified')
], loc='upper right')

plt.tight_layout()
plt.savefig('exp1_elo_recovery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp1_elo_recovery.png')

---
# Experiment 2: Strategy-ELO Interaction Effect
**Hypothesis:** Strategy injection (Aggressive vs Positional) produces greater stylistic differentiation at higher ELO than at lower ELO, because low-ELO vocabulary constraints leave little room for stylistic variation.

**Design:** Generate Aggressive and Positional explanations at all four ELO levels. Compute TF-IDF cosine similarity between them at each ELO. Low similarity = high differentiation. 

**Expected finding:** Similarity should be high (close to 1.0) at ELO 800 — both strategies converge to the same simple language. Similarity should decrease at higher ELO as vocabulary and framing diverge.

In [ ]:
print('Generating Aggressive and Positional explanations across ELO levels...')
exp2_explanations = {'Aggressive': {}, 'Positional': {}}

for strategy in ['Aggressive', 'Positional']:
    for elo in ELO_LEVELS:
        print(f'  {strategy} | ELO {elo}...', end=' ', flush=True)
        text = get_move_explanation(
            board=board, move_san=MOVE_SAN, move_uci=MOVE_UCI,
            elo=elo, evaluation=EVAL_STR, strategy=strategy
        )
        exp2_explanations[strategy][elo] = text
        print(f'done ({len(text.split())} words)')

print('\nAll 8 explanations generated.')

In [ ]:
# Compute TF-IDF cosine similarity between Aggressive and Positional at each ELO
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

# Fit on all 8 texts together for a shared vocabulary
all_texts = [
    exp2_explanations[s][e]
    for s in ['Aggressive', 'Positional']
    for e in ELO_LEVELS
]
vectorizer.fit(all_texts)

interaction_rows = []
for elo in ELO_LEVELS:
    agg_vec  = vectorizer.transform([exp2_explanations['Aggressive'][elo]])
    pos_vec  = vectorizer.transform([exp2_explanations['Positional'][elo]])
    sim      = cosine_similarity(agg_vec, pos_vec)[0][0]
    interaction_rows.append({
        'ELO': elo,
        'ELO Label': ELO_LABELS[elo],
        'Cosine Similarity': round(float(sim), 3),
        'Differentiation': round(1 - float(sim), 3),
    })

df_interaction = pd.DataFrame(interaction_rows)
print('STRATEGY-ELO INTERACTION — COSINE SIMILARITY (Aggressive vs Positional)')
print('='*65)
print('Lower similarity = more stylistic differentiation between strategies')
print()
print(df_interaction[['ELO Label', 'Cosine Similarity', 'Differentiation']].to_string(index=False))
print()

trend = 'decreasing' if df_interaction['Cosine Similarity'].iloc[-1] < df_interaction['Cosine Similarity'].iloc[0] else 'increasing'
print(f'Similarity trend across ELO: {trend}')
print('(Decreasing = strategy differentiation grows with ELO, confirming hypothesis)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Strategy-ELO Interaction Effect\n(Aggressive vs Positional explanation similarity across ELO bands)',
             fontsize=12, fontweight='bold')

elos = df_interaction['ELO'].tolist()
sims = df_interaction['Cosine Similarity'].tolist()
diffs = df_interaction['Differentiation'].tolist()

# Similarity line
axes[0].plot(elos, sims, marker='o', linewidth=2.5, markersize=9, color='#e74c3c')
for x, y in zip(elos, sims):
    axes[0].annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                     xytext=(0, 12), ha='center', fontsize=10, fontweight='bold', color='#e74c3c')
axes[0].set_xlabel('Player ELO', fontsize=11)
axes[0].set_ylabel('Cosine Similarity', fontsize=11)
axes[0].set_title('Similarity (lower = more differentiated)', fontweight='bold')
axes[0].set_xticks(elos)
axes[0].set_xticklabels([str(e) for e in elos])
axes[0].set_ylim(0, 1.1)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.4, label='Maximum similarity')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend()

# Differentiation bars
bar_colors = ['#3498db' if d == max(diffs) else '#95a5a6' for d in diffs]
bars = axes[1].bar([str(e) for e in elos], diffs, color=bar_colors, width=0.5)
for bar, val in zip(bars, diffs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_xlabel('Player ELO', fontsize=11)
axes[1].set_ylabel('Stylistic Differentiation (1 - similarity)', fontsize=11)
axes[1].set_title('Differentiation by ELO\n(blue = highest differentiation)', fontweight='bold')
axes[1].set_ylim(0, 0.8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('exp2_strategy_interaction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp2_strategy_interaction.png')

In [ ]:
# Print side-by-side at most differentiated ELO
best_elo = df_interaction.loc[df_interaction['Differentiation'].idxmax(), 'ELO']
print(f'Most differentiated ELO: {best_elo}')
print(f'\n--- AGGRESSIVE at ELO {best_elo} ---')
print(exp2_explanations['Aggressive'][best_elo])
print(f'\n--- POSITIONAL at ELO {best_elo} ---')
print(exp2_explanations['Positional'][best_elo])

---
# Experiment 3: Positional Complexity vs Explanation Faithfulness
**Hypothesis:** Explanation faithfulness degrades as positions become more complex. A system that handles standard middlegames well may produce vague or inaccurate explanations in sharp tactical or complex endgame positions.

**Design:** Three position types — opening (low complexity), middlegame (moderate), tactical endgame (high complexity). At each position, explanations are generated at all four ELO levels. Faithfulness is measured by: (1) whether the explanation mentions the correct pieces involved in the move, and (2) whether it correctly identifies the key threat or purpose of the move.

**Faithfulness scoring:** Automated proxy — count how many of the key position-specific terms (piece names, square names, tactical motifs present in the position) appear in the explanation. Normalise by expected count.

In [ ]:
# Three test positions with ground truth metadata
POSITIONS = {
    'Opening': {
        'fen':       'rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2',
        'move_san':  'Nc6',
        'move_uci':  'b8c6',
        'eval':      '+0.00',
        'complexity': 'Low',
        'key_terms': ['knight', 'center', 'e5', 'development', 'c6', 'd4'],
        'key_purpose': 'Develops the knight while defending the e5 pawn and controlling central squares'
    },
    'Middlegame': {
        'fen':       'r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/2N2N2/PPPP1PPP/R1BQK2R w KQkq - 4 5',
        'move_san':  'O-O',
        'move_uci':  'e1g1',
        'eval':      '+0.19',
        'complexity': 'Moderate',
        'key_terms': ['castle', 'king', 'safety', 'rook', 'development', 'h1', 'center'],
        'key_purpose': 'Castles to improve king safety and activate the h1 rook'
    },
    'Endgame': {
        'fen':       '8/5pk1/6p1/R7/5PKP/8/8/r7 w - - 0 1',
        'move_san':  'Ra7',
        'move_uci':  'a5a7',
        'eval':      '+1.40',
        'complexity': 'High',
        'key_terms': ['rook', 'seventh', '7th', 'king', 'pawn', 'f7', 'passive', 'active'],
        'key_purpose': 'Activates the rook to the seventh rank to attack pawns and restrict the enemy king'
    }
}

# Display all three positions
for pos_name, pos in POSITIONS.items():
    b = chess.Board(pos['fen'])
    print(f"--- {pos_name} ({pos['complexity']} complexity) ---")
    print(f"Move: {pos['move_san']} | Eval: {pos['eval']}")
    print(f"Purpose: {pos['key_purpose']}")
    display(SVG(chess.svg.board(b, size=250)))
    print()

In [ ]:
# Generate explanations: 3 positions × 4 ELO = 12 calls
print('Generating explanations across position types and ELO levels...')
exp3_explanations = {pos: {} for pos in POSITIONS}

for pos_name, pos in POSITIONS.items():
    b = chess.Board(pos['fen'])
    for elo in ELO_LEVELS:
        print(f'  {pos_name} | ELO {elo}...', end=' ', flush=True)
        text = get_move_explanation(
            board=b,
            move_san=pos['move_san'],
            move_uci=pos['move_uci'],
            elo=elo,
            evaluation=pos['eval'],
            strategy='Balanced'
        )
        exp3_explanations[pos_name][elo] = text
        print(f'done ({len(text.split())} words)')

print('\nAll 12 explanations generated.')

In [ ]:
def faithfulness_score(explanation, key_terms):
    """
    Automated faithfulness proxy:
    What fraction of the position-specific key terms appear in the explanation?
    Higher = more faithful to the actual position.
    """
    text_lower = explanation.lower()
    found = [t for t in key_terms if t.lower() in text_lower]
    score = len(found) / len(key_terms)
    return round(score, 3), found


exp3_rows = []
for pos_name, pos in POSITIONS.items():
    for elo in ELO_LEVELS:
        text = exp3_explanations[pos_name][elo]
        score, found = faithfulness_score(text, pos['key_terms'])
        exp3_rows.append({
            'Position':         pos_name,
            'Complexity':       pos['complexity'],
            'ELO':              elo,
            'ELO Label':        ELO_LABELS[elo],
            'Faithfulness':     score,
            'Terms Found':      ', '.join(found),
            'Terms Missed':     ', '.join(t for t in pos['key_terms'] if t.lower() not in text.lower()),
            'Word Count':       len(text.split()),
        })

df_complexity = pd.DataFrame(exp3_rows)

print('FAITHFULNESS SCORES BY POSITION TYPE AND ELO')
print('='*70)
print('(1.0 = all key terms present, 0.0 = none present)')
print()
pivot = df_complexity.pivot_table(
    index='Position', columns='ELO', values='Faithfulness', aggfunc='mean'
).round(3)
print(pivot.to_string())
print()

avg_by_complexity = df_complexity.groupby('Complexity')['Faithfulness'].mean().round(3)
print('Average faithfulness by complexity:')
print(avg_by_complexity.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Positional Complexity vs Explanation Faithfulness',
             fontsize=13, fontweight='bold')

pos_colors = {'Opening': '#2ecc71', 'Middlegame': '#3498db', 'Endgame': '#e74c3c'}

# Chart 1: Faithfulness by ELO for each position type
for pos_name in POSITIONS:
    subset = df_complexity[df_complexity['Position'] == pos_name].sort_values('ELO')
    axes[0].plot(
        subset['ELO'], subset['Faithfulness'],
        marker='o', linewidth=2.5, markersize=8,
        label=f"{pos_name} ({POSITIONS[pos_name]['complexity']})",
        color=pos_colors[pos_name]
    )
    for _, row in subset.iterrows():
        axes[0].annotate(
            f"{row['Faithfulness']:.2f}",
            (row['ELO'], row['Faithfulness']),
            textcoords='offset points', xytext=(0, 10),
            ha='center', fontsize=8, color=pos_colors[pos_name]
        )

axes[0].set_xlabel('Player ELO', fontsize=11)
axes[0].set_ylabel('Faithfulness Score', fontsize=11)
axes[0].set_title('Faithfulness by Position Type and ELO', fontweight='bold')
axes[0].set_xticks(ELO_LEVELS)
axes[0].set_xticklabels([str(e) for e in ELO_LEVELS])
axes[0].set_ylim(0, 1.2)
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Chart 2: Average faithfulness by complexity
complexity_order = ['Low', 'Moderate', 'High']
avg_scores = [avg_by_complexity.get(c, 0) for c in complexity_order]
bar_colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = axes[1].bar(complexity_order, avg_scores, color=bar_colors, width=0.5)
for bar, val in zip(bars, avg_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Position Complexity', fontsize=11)
axes[1].set_ylabel('Avg Faithfulness Score', fontsize=11)
axes[1].set_title('Faithfulness Degradation by Complexity\n(Lower = system struggles more)',
                   fontweight='bold')
axes[1].set_ylim(0, 1.2)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('exp3_positional_complexity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp3_positional_complexity.png')

In [ ]:
# Print missed terms per position to show where system fails
print('TERMS MISSED BY POSITION AND ELO')
print('='*70)
for pos_name in POSITIONS:
    print(f'\n{pos_name}:')
    subset = df_complexity[df_complexity['Position'] == pos_name]
    for _, row in subset.iterrows():
        missed = row['Terms Missed'] if row['Terms Missed'] else 'none'
        print(f'  ELO {row["ELO"]:4d}: faithfulness={row["Faithfulness"]:.2f} | missed: {missed}')

---
# Combined Summary

In [ ]:
print('='*70)
print('EXPERIMENT SUMMARY')
print('='*70)

print(f"""
EXPERIMENT 1 — ELO Recovery
  Accuracy: {accuracy:.0%}
  Mean ELO error on misclassifications: {df_recovery['ELO Error'].mean():.0f} points
  Finding: {'Explanations are semantically distinct enough to recover ELO with high accuracy.' if accuracy >= 0.75 else 'Partial semantic differentiation — surface metrics outperform semantic ones.'}

EXPERIMENT 2 — Strategy-ELO Interaction
  Similarity range: {df_interaction['Cosine Similarity'].max():.3f} (ELO {df_interaction.loc[df_interaction['Cosine Similarity'].idxmax(),'ELO']}) → {df_interaction['Cosine Similarity'].min():.3f} (ELO {df_interaction.loc[df_interaction['Cosine Similarity'].idxmin(),'ELO']})
  Most differentiated at: ELO {best_elo}
  Finding: {'Strategy differentiation increases with ELO, confirming interaction hypothesis.' if df_interaction['Cosine Similarity'].iloc[-1] < df_interaction['Cosine Similarity'].iloc[0] else 'Strategy differentiation does not consistently increase with ELO — hypothesis not confirmed.'}

EXPERIMENT 3 — Positional Complexity
  Opening faithfulness:    {avg_by_complexity.get('Low', 0):.3f}
  Middlegame faithfulness: {avg_by_complexity.get('Moderate', 0):.3f}
  Endgame faithfulness:    {avg_by_complexity.get('High', 0):.3f}
  Finding: {'Faithfulness degrades as position complexity increases, indicating a system limitation on tactical/endgame positions.' if avg_by_complexity.get('High',1) < avg_by_complexity.get('Low',0) else 'Faithfulness holds across complexity levels.'}
""")